# Following the shadow axis across Earth

Take the eclipse of 8 April 2024. At what point on Earth would an observer see the centers of the Sun and Moon coincide? The answer changes as the Moon moves and Earth rotates. If we can find the point at two nearby instants, we can also estimate its speed across the ground.

This calculation uses the NASA reference coordinates in `dataset_C.csv` (in this folder or in `data/`) and downloads the three barycentric states from Horizons on the first run. Its output is a central line. A partial eclipse can be visible even when this line misses Earth.

In [ ]:
%pip install numpy pandas scipy astropy pyproj matplotlib

In [ ]:
from datetime import date as Date
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen
import csv
import io
import numpy as np
import pandas as pd
from scipy.interpolate import CubicHermiteSpline
from astropy.time import Time, TimeDelta
from astropy.coordinates import GCRS, ITRS, CartesianRepresentation, EarthLocation
import astropy.units as u
from pyproj import Geod

LIGHT_SPEED_KMS = 299792.458
CACHE_DIR = Path('cache')
CACHE_DIR.mkdir(exist_ok=True)
GEOD = Geod(ellps='WGS84')
MONTHS = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
         'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']


SUN_RADIUS_KM = 695700.0
MOON_RADIUS_KM = 1737.4
EQUATORIAL_RADIUS_KM = 6378.137
FLATTENING = 1 / 298.257223563
POLAR_RADIUS_KM = EQUATORIAL_RADIUS_KM * (1 - FLATTENING)

## Attaching axes to Earth

Let $B(t)$ contain the terrestrial basis vectors expressed in the celestial frame. Then $\mathbf r_T=B^\top\mathbf r$ and $\mathbf v_T=B^\top\mathbf v+\dot B^\top\mathbf r$. The second term accounts for the rotation of the axes. We approximate $\dot B$ with a centered difference, taking the time step in seconds.

In [ ]:
def earth_basis(t):

    itrs_axes = ITRS(CartesianRepresentation(np.eye(3) * u.km), obstime=t)
    gcrs_axes = itrs_axes.transform_to(GCRS(obstime=t))
    return gcrs_axes.cartesian.xyz.to_value(u.km)


def rotate_state(t, r, v, h=1.0):

    B = earth_basis(t)
    delta = TimeDelta(h, format='sec')
    basis_derivative = (earth_basis(t + delta) - earth_basis(t - delta)) / (2 * h)

    r = np.asarray(r, dtype=float)
    v = np.asarray(v, dtype=float)
    r_itrs = B.T @ r
    v_itrs = B.T @ v + basis_derivative.T @ r
    return r_itrs, v_itrs

## Where does the axis meet the surface?

The ray leaves the Moon away from the Sun. Write $P(\lambda)=\mathbf r_M+\lambda\hat{\mathbf d}$ with $\lambda\geq0$ in km. Substitution into $(x^2+y^2)/a^2+z^2/b^2=1$ gives a quadratic in $\lambda$. The smaller forward root is the surface first reached by the shadow. At that point the centers coincide. Comparing $\arcsin(R_M/d_M)$ and $\arcsin(R_S/d_S)$ distinguishes totality from annularity.

In [ ]:
def central_point(rS, rL):

    rS = np.asarray(rS, dtype=float)
    rL = np.asarray(rL, dtype=float)


    d = rL - rS
    d = d / np.linalg.norm(d)


    a = EQUATORIAL_RADIUS_KM
    b = POLAR_RADIUS_KM
    A = (d[0]**2 + d[1]**2) / a**2 + d[2]**2 / b**2
    B = 2 * ((rL[0]*d[0] + rL[1]*d[1]) / a**2 + rL[2]*d[2] / b**2)
    C = (rL[0]**2 + rL[1]**2) / a**2 + rL[2]**2 / b**2 - 1
    discriminant = B**2 - 4*A*C

    if discriminant < 0:
        return None, 'no_central_line'

    root = np.sqrt(discriminant)
    roots = [(-B - root) / (2*A), (-B + root) / (2*A)]
    roots = [value for value in roots if value >= 0]
    if not roots:
        return None, 'no_central_line'

    point = rL + min(roots) * d


    moon_distance = np.linalg.norm(point - rL)
    sun_distance = np.linalg.norm(point - rS)
    moon_radius_rad = np.arcsin(np.clip(MOON_RADIUS_KM / moon_distance, 0, 1))
    sun_radius_rad = np.arcsin(np.clip(SUN_RADIUS_KM / sun_distance, 0, 1))
    kind = 'total' if moon_radius_rad >= sun_radius_rad else 'annular'
    return point, kind


def lonlat(point):

    x, y, z = np.asarray(point, dtype=float)
    location = EarthLocation.from_geocentric(x, y, z, unit=u.km)
    lon, lat, _ = location.to_geodetic(ellipsoid='WGS84')
    return float(lon.deg), float(lat.deg)

## Sampling the orbit

Five-minute barycentric positions and velocities come from Horizons in TDB. The query starts before midnight so that light received early in the day can have left the Sun on the preceding day.

In [ ]:
def normalize_date(date_string):
    return Date.fromisoformat(date_string).isoformat()





def fetch_ephemerides(date_string):
    cache_file = CACHE_DIR / f'JPL_barycentric_5min_TDB_{date_string}.csv'
    legacy_file = CACHE_DIR / f"JPL_baricentrico_5min_TDB_{date_string}.csv"
    if not cache_file.exists() and legacy_file.exists():
        return pd.read_csv(legacy_file).rename(columns={"segundos": "seconds"})
    if cache_file.exists():
        return pd.read_csv(cache_file)

    start_time = Time(date_string, scale='utc')
    query_start = (start_time - TimeDelta(900, format='sec')).tdb.iso
    query_end = (start_time + TimeDelta(86400 + 300, format='sec')).tdb.iso

    bodies = {'earth': '399', 'moon': '301', 'sun': '10'}
    df = None

    for body, body_id in bodies.items():
        params = {
            'format': 'text', 'COMMAND': f"'{body_id}'",
            'CENTER': "'500@0'", 'MAKE_EPHEM': "'YES'",
            'OBJ_DATA': "'NO'", 'EPHEM_TYPE': "'VECTORS'",
            'REF_SYSTEM': "'ICRF'", 'REF_PLANE': "'FRAME'",
            'VEC_CORR': "'NONE'", 'TIME_TYPE': "'TDB'",
            'OUT_UNITS': "'KM-S'", 'VEC_TABLE': "'2'",
            'CSV_FORMAT': "'YES'", 'VEC_LABELS': "'YES'",
            'START_TIME': f"'{query_start}'",
            'STOP_TIME': f"'{query_end}'", 'STEP_SIZE': "'5 m'"
        }

        raw_file = CACHE_DIR / f'JPL_barycentric_5min_TDB_{date_string}_{body}.txt'
        if raw_file.exists():
            response = raw_file.read_text(encoding='utf-8')
        else:
            url = 'https://ssd.jpl.nasa.gov/api/horizons.api?' + urlencode(params)
            response = urlopen(url, timeout=60).read().decode('utf-8')
            if '$$SOE' not in response or '$$EOE' not in response:
                raise ValueError(f'Horizons error for {body}: {response[:300]}')
            raw_file.write_text(response, encoding='utf-8')

        rows = list(csv.reader(io.StringIO(
            response.split('$$SOE')[1].split('$$EOE')[0].strip()
        )))
        data = np.array([[float(row[i]) for i in (0, 2, 3, 4, 5, 6, 7)]
                          for row in rows])

        times = Time(data[:, 0], format='jd', scale='tdb')
        seconds = (times.tai - start_time.tai).to_value('s')

        if df is None:
            df = pd.DataFrame({'seconds': seconds})
        elif not np.allclose(df['seconds'], seconds, rtol=0, atol=0.002):
            raise ValueError('The timestamps of the three bodies differ.')

        columns = [f'{variable}_{body}' for variable in
                    ('x_km', 'y_km', 'z_km', 'vx_kms', 'vy_kms', 'vz_kms')]
        df[columns] = data[:, 1:]

    df.to_csv(cache_file, index=False)
    return df

Consider light arriving at Earth at $t$. It passed the Moon at $t_M=t-\tau_M$ and left the Sun at $t_S=t_M-\tau_{SM}$. We estimate each delay from distance divided by $c$ and update it twice using the earlier body position. Subtracting Earth's position at reception gives the geocentric vectors. A cubic Hermite spline matches both position and velocity at each five-minute sample, allowing evaluation at these intermediate times.

In [ ]:
def transform_frame(date_string, df_jpl):
    ephemerides = {}
    for body in ('earth', 'moon', 'sun'):
        position = df_jpl[[f'{e}_{body}' for e in ('x_km', 'y_km', 'z_km')]]
        velocity = df_jpl[[f'{e}_{body}' for e in ('vx_kms', 'vy_kms', 'vz_kms')]]
        ephemerides[body] = CubicHermiteSpline(
            df_jpl['seconds'], position.to_numpy(), velocity.to_numpy(),
            extrapolate=False
        )

    earth = ephemerides['earth']
    moon = ephemerides['moon']
    sun = ephemerides['sun']
    rows = []
    start_time = Time(date_string, scale='utc')

    for s in np.arange(0, 86401, 300):
        RE = earth(s)


        tau_L = np.linalg.norm(moon(s) - RE) / LIGHT_SPEED_KMS
        for _ in range(2):
            tau_L = np.linalg.norm(moon(s - tau_L) - RE) / LIGHT_SPEED_KMS
        tL = s - tau_L


        tau_SL = np.linalg.norm(sun(tL) - moon(tL)) / LIGHT_SPEED_KMS
        for _ in range(2):
            tau_SL = np.linalg.norm(sun(tL - tau_SL) - moon(tL)) / LIGHT_SPEED_KMS
        tS = tL - tau_SL


        rL = moon(tL) - RE
        rS = sun(tS) - RE


        vL = moon.derivative()(tL) - earth.derivative()(s)
        vS = sun.derivative()(tS) - earth.derivative()(s)

        t = start_time + TimeDelta(float(s), format='sec')
        rL_itrs, vL_itrs = rotate_state(t, rL, vL)
        rS_itrs, vS_itrs = rotate_state(t, rS, vS)

        row = {'seconds': s, 'utc': t.utc.isot,
                'tau_L_s': tau_L, 'tau_SL_s': tau_SL}
        for body, r, v in [('moon', rL_itrs, vL_itrs),
                             ('sun', rS_itrs, vS_itrs)]:
            for coord, value in zip(('x', 'y', 'z'), r):
                row[f'{coord}_km_{body}'] = value
            for coord, value in zip(('vx', 'vy', 'vz'), v):
                row[f'{coord}_kms_{body}'] = value
        rows.append(row)

    df_itrs = pd.DataFrame(rows)


    interpolators = {}
    for body in ('moon', 'sun'):
        r = df_itrs[[f'{e}_km_{body}' for e in ('x', 'y', 'z')]]
        v = df_itrs[[f'{e}_kms_{body}' for e in ('vx', 'vy', 'vz')]]
        interpolators[body] = CubicHermiteSpline(
            df_itrs['seconds'], r.to_numpy(), v.to_numpy(), extrapolate=False
        )
    return df_itrs, interpolators

## The moving point

For each sample we keep the Cartesian intersection in km and its WGS84 geodetic latitude and longitude in degrees. We estimate its velocity from two neighboring intersections and project onto the local east and north unit vectors. Near first and last contact one neighbor may be absent, leaving the velocity undefined.

In [ ]:
def geometry(df_itrs, interpolators):
    rows = []
    for s, utc in zip(df_itrs['seconds'], df_itrs['utc']):
        rS = interpolators['sun'](s)
        rL = interpolators['moon'](s)
        point, kind = central_point(rS, rL)

        row = {'seconds': s, 'utc': utc, 'kind': kind,
                'lat': np.nan, 'lon': np.nan,
                'x': np.nan, 'y': np.nan, 'z': np.nan}
        if point is not None:
            lon, lat = lonlat(point)
            row.update(lat=lat, lon=lon,
                        x=point[0], y=point[1], z=point[2])
        rows.append(row)
    return pd.DataFrame(rows)


def dynamics(geometry_df, interpolators, h=1.0):
    df = geometry_df.copy()
    df['east_velocity_kms'] = np.nan
    df['north_velocity_kms'] = np.nan
    df['speed_kms'] = np.nan

    for i, row in df.dropna(subset=['lat', 'lon']).iterrows():
        s = row['seconds']
        a, b = max(0, s - h), min(86400, s + h)

        before, _ = central_point(interpolators['sun'](a),
                                    interpolators['moon'](a))
        after, _ = central_point(interpolators['sun'](b),
                                     interpolators['moon'](b))
        if before is None or after is None:
            continue

        v = (after - before) / (b - a)
        lon, lat = np.deg2rad([row['lon'], row['lat']])
        east = np.array([-np.sin(lon), np.cos(lon), 0])
        north = np.array([-np.sin(lat)*np.cos(lon),
                          -np.sin(lat)*np.sin(lon), np.cos(lat)])

        df.loc[i, 'east_velocity_kms'] = np.dot(v, east)
        df.loc[i, 'north_velocity_kms'] = np.dot(v, north)
        df.loc[i, 'speed_kms'] = np.linalg.norm(v)

    return df

## Comparing the same instant

Each timed NASA central coordinate is paired with the calculated intersection at its UT1 timestamp. The WGS84 geodesic gives a distance in km. Dividing by the tabulated path width expresses the displacement as a percentage of the central band.

In [ ]:
def compare_reference(date_string, interpolators, df_C):
    d = Date.fromisoformat(date_string)
    event = f'SE{d.year}{MONTHS[d.month-1]}{d.day:02d}'
    reference = df_C[df_C['event_id'].str.startswith(event, na=False)]
    reference = reference.dropna(subset=['center_lat', 'center_lon'])
    start_time = Time(date_string, scale='utc')
    rows = []

    for _, record in reference.iterrows():
        clock = record['time_ut']
        if clock.count(':') == 1:
            clock += ':00'
        t = Time(f'{date_string} {clock}', scale='ut1')
        s = (t.tai - start_time.tai).to_value('s')
        if not 0 <= s <= 86400:
            continue

        point, _ = central_point(interpolators['sun'](s),
                                 interpolators['moon'](s))
        if point is None:
            continue
        lon, lat = lonlat(point)
        _, _, distance_m = GEOD.inv(lon, lat,
                                      record['center_lon'], record['center_lat'])
        error_km = distance_m / 1000
        width = record['path_width_km']
        percent = 100 * error_km / width if pd.notna(width) and width > 0 else np.nan
        rows.append({'reference_ut1': record['time_ut'], 'model_lat': lat,
                      'model_lon': lon, 'lat_nasa': record['center_lat'],
                      'lon_nasa': record['center_lon'], 'error_km': error_km,
                      'error_percent_width': percent})

    errors = pd.DataFrame(rows)
    if not errors.empty:
        print('Matched points:', len(errors))
        print('Median error (km):', round(errors['error_km'].median(), 3))
        print('Mean error (km):', round(errors['error_km'].mean(), 3))
        print('Maximum error (km):', round(errors['error_km'].max(), 3))
        print('Median error as a fraction of path width (%):',
              round(errors['error_percent_width'].median(), 3))
    else:
        print('No reference central points are available.')
    return errors





def run(input_date):
    date_string = normalize_date(input_date)


    df_jpl = fetch_ephemerides(date_string)
    df_itrs, interpolators = transform_frame(date_string, df_jpl)
    geometry_df = geometry(df_itrs, interpolators)
    dynamics_df = dynamics(geometry_df, interpolators)


    reference_file = Path('data/dataset_C.csv')
    if not reference_file.exists():
        reference_file = Path('dataset_C.csv')
    df_C = pd.read_csv(reference_file)
    errors_df = compare_reference(date_string, interpolators, df_C)


    return {'jpl': df_jpl, 'itrs': df_itrs,
            'geometry': geometry_df, 'dynamics': dynamics_df,
            'errors': errors_df}

In [ ]:
event_date = "2024-04-08"
result = run(event_date)

For a comparison across the labeled dates, the optional batch below runs the same physical model on every positive reference day in `df_jpl_complete.csv`. The plots that follow use the April 2024 result selected above.

In [ ]:
RUN_BATCH = False
if RUN_BATCH:
    observations = pd.read_csv("df_jpl_complete.csv")
    dates = observations.loc[observations["eclipse"] == 1, "reference_day"].unique()
    for batch_date in dates:
        run(batch_date)

The reference table records a clock reading. Attach the event date before converting UT1 to UTC for the horizontal axis. In the recorded April 2024 comparison, 97 matched points gave a median displacement of 1.717 km and a maximum of 17.485 km. The time series shows where those larger displacements occur.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from datetime import timezone

df = result["errors"].copy()

df["reference_ut1"] = pd.to_datetime(Time([f"{event_date} {clock}" for clock in df["reference_ut1"]], scale="ut1").utc.isot, utc=True)
df = df.dropna(subset=["reference_ut1", "error_km"])
df = df.sort_values("reference_ut1")

median = df["error_km"].median()
mean = df["error_km"].mean()
maximum = df.loc[df["error_km"].idxmax()]

plt.figure(figsize=(12, 5))

plt.plot(
    df["reference_ut1"],
    df["error_km"],
    marker="o",
    markersize=3,
    label="Physical model"
)

plt.axhline(
    median,
    color="green",
    linestyle="--",
    label=f"Median = {median:.3f} km"
)

plt.axhline(
    mean,
    color="orange",
    linestyle="--",
    label=f"Mean = {mean:.3f} km"
)

plt.scatter(
    maximum["reference_ut1"],
    maximum["error_km"],
    color="red",
    s=70,
    zorder=5,
    label=f"Maximum = {maximum['error_km']:.3f} km"
)

plt.gca().xaxis.set_major_formatter(
    mdates.DateFormatter("%H:%M", tz=timezone.utc)
)

plt.xlabel("Time (UTC)")
plt.ylabel("Geodesic error (km)")
plt.title("Distance from the NASA central line")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


display(
    df.nlargest(5, "error_km")[
        ["reference_ut1", "error_km", "error_percent_width"]
    ]
)

## Why are the endpoints sensitive?

At the intersection, the outward normal is parallel to $(x/a^2,y/a^2,z/b^2)$. Let $\theta$ be the acute angle between the axis and that normal. A displacement along the ray has only the fraction $|\cos\theta|$ in the normal direction. To make up a given normal displacement, the intersection therefore travels a distance proportional to $K=1/|\cos\theta|$. When the ray grazes Earth, $K$ grows rapidly. We compare its rank with the rank of the geodesic error along this one eclipse.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from scipy.interpolate import CubicHermiteSpline
from scipy.stats import spearmanr
from astropy.time import Time


date_string = event_date

df = result["errors"].copy()
itrs = result["itrs"].copy()

start_time = Time(date_string, scale="utc")


interpolators = {}

for body in ("sun", "moon"):

    r = itrs[
        [f"{e}_km_{body}" for e in ("x", "y", "z")]
    ].to_numpy()

    v = itrs[
        [f"{e}_kms_{body}" for e in ("vx", "vy", "vz")]
    ].to_numpy()

    interpolators[body] = CubicHermiteSpline(
        itrs["seconds"].to_numpy(),
        r,
        v,
        extrapolate=False
    )


a = 6378.137
b = a * (1 - 1 / 298.257223563)


angles = []
sensitivities = []
times = []
seconds = []

for _, row in df.iterrows():


    clock = str(row["reference_ut1"]).split(" ")[-1]
    clock = clock.split("T")[-1][:8]

    if len(clock) == 5:
        clock += ":00"


    t = Time(f"{date_string} {clock}", scale="ut1")
    s = (t.tai - start_time.tai).to_value("s")

    if not itrs["seconds"].min() <= s <= itrs["seconds"].max():
        raise ValueError(f"Time outside the interpolation interval: {clock}")


    rS = interpolators["sun"](s)
    rL = interpolators["moon"](s)


    d = rL - rS
    d = d / np.linalg.norm(d)


    point, kind = central_point(rS, rL)

    if point is None:
        angles.append(np.nan)
        sensitivities.append(np.nan)
    else:

        x, y, z = point


        n = np.array([
            x / a**2,
            y / a**2,
            z / b**2
        ])

        n = n / np.linalg.norm(n)

        cos_theta = abs(np.dot(d, n))
        cos_theta = np.clip(cos_theta, 0, 1)

        theta = np.degrees(np.arccos(cos_theta))
        K = 1 / cos_theta if cos_theta > 0 else np.inf

        angles.append(theta)
        sensitivities.append(K)

    times.append(pd.Timestamp(t.utc.isot, tz="UTC"))
    seconds.append(s)

df["reference_utc"] = times
df["seconds"] = seconds
df["theta_deg"] = angles
df["K"] = sensitivities

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=["error_km", "K"])
df = df.sort_values("reference_utc")


fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(12, 8), sharex=True
)

ax1.plot(
    df["reference_utc"],
    df["error_km"],
    marker=".",
    label="Distance from NASA reference"
)
ax1.set_ylabel("Error (km)")
ax1.set_title("Central-line error")
ax1.grid(alpha=0.3)
ax1.legend()

ax2.plot(
    df["reference_utc"],
    df["K"],
    color="darkorange",
    marker=".",
    label="K = 1 / |cos(theta)|"
)
ax2.set_ylabel("Geometric sensitivity K")
ax2.set_xlabel("Time (UTC)")
ax2.grid(alpha=0.3)
ax2.legend()

ax2.xaxis.set_major_formatter(
    mdates.DateFormatter("%H:%M", tz="UTC")
)

plt.tight_layout()
plt.show()


rho, p_value = spearmanr(df["K"], df["error_km"])

plt.figure(figsize=(8, 5))

plt.scatter(
    df["K"],
    df["error_km"],
    alpha=0.7
)

plt.xlabel("Geometric sensitivity K")
plt.ylabel("Distance from NASA reference (km)")
plt.title(f"Error versus sensitivity (Spearman = {rho:.3f})")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


print("First UTC instant:", df["reference_utc"].iloc[0])
print("Last UTC instant:", df["reference_utc"].iloc[-1])
print("Spearman correlation:", round(rho, 4))
print("p-value:", p_value)
print("Minimum sensitivity:", round(df["K"].min(), 4))
print("Maximum sensitivity:", round(df["K"].max(), 4))

display(
    df.nlargest(10, "error_km")[
        ["reference_utc", "error_km", "theta_deg", "K"]
    ]
)

For the recorded track the Spearman correlation was 0.9624. This is evidence that large errors cluster near grazing intersections along this path. It gives neither the direction of a correction nor a correction in kilometres. Those are the quantities tested in Model 2.